In [1]:
#| default_exp restxl

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [5]:
#| export
from rest.core import init_instance, process_seq
singleton, model_path = init_instance()

In [6]:
#| export
import torch
import regex as re

In [7]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length
)

[W socket.cpp:426] [c10d] The server socket cannot be initialized on [::]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).


> initializing model parallel with size 1
Use alternating sparse & dense attention layers


In [8]:
#| export
tokenizer = model.tokenizer
model.cuda()
model.eval();

In [9]:
sum(p.numel() for p in model.parameters())

1315737600

In [10]:
#| export
import deepspeed
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.half,
                                 checkpoint=None,
                                 replace_method='auto',
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2022-11-11 12:02:01,216] [INFO] [logging.py:68:log_dist] [Rank -1] DeepSpeed info: version=0.7.5+28d4fdb, git-hash=28d4fdb, git-branch=master
[2022-11-11 12:02:01,217] [INFO] [logging.py:68:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [11]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    bad_words_ids = [tokenizer.encode('[')[0], tokenizer.encode('(')[0], tokenizer.encode('1\xa01')[1]]
    linebreak = tokenizer.encode("1\n1")[1]
    lb2 = tokenizer.encode("1 \n")[1]
    bad_words_ids += [] if allow_linebreak else [linebreak, lb2]
    bad_words_ids = [[b] for b in bad_words_ids] + [[linebreak,linebreak]]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length=length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            bad_words_ids = bad_words_ids
        )
    generated_sequences = output_sequences
    return process_seq(generated_sequences)

In [12]:
%%time
get_sample(' - ты кто?', 50, 4, False)

not setting adaptive thresholding
CPU times: user 17 s, sys: 833 ms, total: 17.8 s
Wall time: 9.39 s


[' - ты кто? Перезагрузка. Вы - дети одной семьи, а значит связаны воспоминаниями о пережитых вместе днях. Вам многое предстоит пережить. Но эти минуты есть только для вас двоих! Никто не сможет помочь, а помогать некому.',
 ' - ты кто? о! тут обрыв!.. но верь: там путь верней!Пей сладкое вино любви, бокал свободы.Я упиваюсь пенным счастьем! Я лечу!..О, дивная моя судьбина!',
 ' - ты кто? Смотрю, а ноги-то - вон они, на кровати. Точно - я, только на ней лет, должно, на двадцать старше. И кстати, папочка - это ты.',
 ' - ты кто? Ты Бог? Ты премудрость, Ты любовь?" Изрыгая зловоние, начала ползти гадюка, подползши к жаровне, гореть. От тела ее пламя распространилось по всему ковру.']

In [24]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

not setting adaptive thresholding
 - ты кто?
Мой ангел, мой избавитель,
мой ангел-хранитель?
Открой мне скорей свой лик
и назовися.

CPU times: user 1min 28s, sys: 2.09 s, total: 1min 30s
Wall time: 1.75 s
